In [0]:
 %pip install scikit-learn plotly pandas numpy mlflow --quiet

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
 
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
 
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA
 
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
 
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Load Data

In [0]:
raw_spark = spark.table("smart_odoo.gold.fact_invoice")
 
print(f"📊 Total rows loaded : {raw_spark.count():,}")
print(f"📋 Total columns     : {len(raw_spark.columns)}")
print()
raw_spark.printSchema()
 
# COMMAND ----------
 
# Convert to Pandas for feature engineering (dataset is partner-level aggregation)
raw_df = raw_spark.toPandas()
 
# Quick sanity check
print("Shape:", raw_df.shape)
display(raw_df.head(5))

##  Exploratory Data Analysis (EDA)

In [0]:
  
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Invoice Overview — Payment Behavior", fontsize=16, fontweight="bold", y=1.02)
 
# Payment State
payment_counts = raw_df["payment_state"].value_counts()
colors_payment = ["#2ecc71", "#e74c3c", "#f39c12"]
axes[0].pie(payment_counts, labels=payment_counts.index, autopct="%1.1f%%",
            colors=colors_payment, startangle=90, wedgeprops=dict(edgecolor="white", linewidth=2))
axes[0].set_title("Payment State Distribution", fontweight="bold")
 
# Invoice Type
type_counts = raw_df["move_type"].value_counts()
axes[1].bar(["Purchase\n(in_invoice)", "Sales\n(out_invoice)"],
            type_counts.values, color=["#3498db", "#9b59b6"], edgecolor="white", linewidth=1.5)
axes[1].set_title("Invoice Type Distribution", fontweight="bold")
axes[1].set_ylabel("Count")
for i, v in enumerate(type_counts.values):
    axes[1].text(i, v + 2, str(v), ha="center", fontweight="bold")
 
# Currency
currency_counts = raw_df["currency_name"].value_counts()
axes[2].bar(currency_counts.index, currency_counts.values,
            color=["#1abc9c", "#e67e22", "#34495e"], edgecolor="white", linewidth=1.5)
axes[2].set_title("Currency Distribution", fontweight="bold")
axes[2].set_ylabel("Count")
for i, v in enumerate(currency_counts.values):
    axes[2].text(i, v + 2, str(v), ha="center", fontweight="bold")
 
plt.tight_layout()
plt.savefig("/tmp/eda_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ EDA Overview plotted")
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC ### 2.2 — توزيع مبالغ الفواتير
 
# COMMAND ----------
 
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Invoice Amount Distribution", fontsize=14, fontweight="bold")
 
# Histogram
axes[0].hist(raw_df["amount_total"], bins=40, color="#3498db", edgecolor="white", alpha=0.8)
axes[0].set_title("Amount Total Distribution (Raw)", fontweight="bold")
axes[0].set_xlabel("Amount Total")
axes[0].set_ylabel("Frequency")
axes[0].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x/1000:.0f}K"))
 
# Log scale
axes[1].hist(np.log1p(raw_df["amount_total"]), bins=40, color="#9b59b6", edgecolor="white", alpha=0.8)
axes[1].set_title("Amount Total Distribution (Log Scale)", fontweight="bold")
axes[1].set_xlabel("log(Amount Total + 1)")
axes[1].set_ylabel("Frequency")
 
plt.tight_layout()
plt.show()
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC ### 2.3 — تحليل سلوك الدفع حسب العميل
 
# COMMAND ----------
 
# Top partners by invoice count
top_partners = (raw_df.groupby("partner_name")
                .agg(invoice_count=("move_id", "count"),
                     total_amount=("amount_total", "sum"),
                     paid_pct=("payment_state", lambda x: (x == "paid").mean() * 100))
                .sort_values("invoice_count", ascending=False)
                .head(15))
 
fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(top_partners.index, top_partners["invoice_count"],
               color=plt.cm.RdYlGn(top_partners["paid_pct"] / 100))
ax.set_xlabel("Number of Invoices")
ax.set_title("Top 15 Partners by Invoice Count\n(Color = % Paid — Green: High, Red: Low)",
             fontweight="bold")
ax.invert_yaxis()
 
# Add value labels
for bar, pct in zip(bars, top_partners["paid_pct"]):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
            f"{bar.get_width():.0f} inv | {pct:.0f}% paid",
            va="center", fontsize=9)
 
plt.tight_layout()
plt.show()
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC ### 2.4 — Correlation Heatmap للمبالغ
 
# COMMAND ----------
 
amount_cols = ["amount_untaxed", "amount_tax", "amount_total", "amount_residual", "invoice_currency_rate"]
corr = raw_df[amount_cols].corr()
 
fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, ax=ax, linewidths=0.5, square=True,
            cbar_kws={"shrink": 0.8})
ax.set_title("Correlation Matrix — Amount Features", fontweight="bold")
plt.tight_layout()
plt.show()

##  Feature Engineering

In [0]:

REFERENCE_DATE = pd.Timestamp("2026-06-17")
 
# ─── Normalize amounts to EGP for fair comparison ─────────────────────────────
# EGP rate = 1, USD ≈ 48.5, EUR ≈ 52 (from data)
CURRENCY_RATES = {"EGP": 1.0, "USD": 48.5, "EUR": 52.0}
 
raw_df["amount_total_egp"] = raw_df.apply(
    lambda r: r["amount_total"] * CURRENCY_RATES.get(r["currency_name"], 1.0), axis=1
)
raw_df["amount_residual_egp"] = raw_df.apply(
    lambda r: r["amount_residual"] * CURRENCY_RATES.get(r["currency_name"], 1.0), axis=1
)
 
# ─── Convert date columns to datetime ─────────────────────────────────────────
raw_df["invoice_date"] = pd.to_datetime(raw_df["invoice_date"])
raw_df["invoice_date_due"] = pd.to_datetime(raw_df["invoice_date_due"])
 
# ─── Payment terms (days) ──────────────────────────────────────────────────────
raw_df["payment_terms_days"] = (raw_df["invoice_date_due"] - raw_df["invoice_date"]).dt.days
 
# ─── Aggregate at Partner Level ────────────────────────────────────────────────
features_df = (
    raw_df.groupby(["partner_id", "partner_name"])
    .agg(
        # Volume
        total_invoices        = ("move_id",            "count"),
        total_amount_egp      = ("amount_total_egp",   "sum"),
        avg_amount_egp        = ("amount_total_egp",   "mean"),
        std_amount_egp        = ("amount_total_egp",   "std"),
        max_amount_egp        = ("amount_total_egp",   "max"),
 
        # Payment Behavior
        paid_count            = ("payment_state",      lambda x: (x == "paid").sum()),
        partial_count         = ("payment_state",      lambda x: (x == "partial").sum()),
        not_paid_count        = ("payment_state",      lambda x: (x == "not_paid").sum()),
 
        # Residual
        total_residual_egp    = ("amount_residual_egp","sum"),
 
        # Terms
        avg_payment_terms     = ("payment_terms_days", "mean"),
 
        # State
        cancel_count          = ("state",              lambda x: (x == "cancel").sum()),
        draft_count           = ("state",              lambda x: (x == "draft").sum()),
        posted_count          = ("state",              lambda x: (x == "posted").sum()),
 
        # Reversal
        has_reversal          = ("reversed_entry_id",  lambda x: int(x.notna().any())),
 
        # Currency diversity
        currency_diversity    = ("currency_name",      "nunique"),
 
        # Invoice type (supplier vs customer)
        invoice_type_mode     = ("move_type",          lambda x: x.mode()[0]),
 
        # Recency & Tenure
        last_invoice_date     = ("invoice_date",       "max"),
        first_invoice_date    = ("invoice_date",       "min"),
 
        # Sent invoices
        sent_ratio_count      = ("is_move_sent",       "sum"),
    )
    .reset_index()
)
 
# ─── Derived Features ──────────────────────────────────────────────────────────
features_df["paid_ratio"]         = features_df["paid_count"]     / features_df["total_invoices"]
features_df["partial_ratio"]      = features_df["partial_count"]  / features_df["total_invoices"]
features_df["overdue_ratio"]      = features_df["not_paid_count"] / features_df["total_invoices"]
features_df["cancel_ratio"]       = features_df["cancel_count"]   / features_df["total_invoices"]
features_df["sent_ratio"]         = features_df["sent_ratio_count"] / features_df["total_invoices"]
 
features_df["collection_rate"]    = 1 - (
    features_df["total_residual_egp"] / features_df["total_amount_egp"].replace(0, np.nan)
).fillna(0).clip(0, 1)
 
features_df["amount_volatility"]  = (
    features_df["std_amount_egp"] / features_df["avg_amount_egp"].replace(0, np.nan)
).fillna(0)
 
features_df["recency_days"]       = (REFERENCE_DATE - features_df["last_invoice_date"]).dt.days
features_df["tenure_days"]        = (features_df["last_invoice_date"] - features_df["first_invoice_date"]).dt.days
 
features_df["is_supplier"]        = (features_df["invoice_type_mode"] == "in_invoice").astype(int)
 
# Log-transform skewed amount features
features_df["log_total_amount"]   = np.log1p(features_df["total_amount_egp"])
features_df["log_avg_amount"]     = np.log1p(features_df["avg_amount_egp"])
features_df["log_max_amount"]     = np.log1p(features_df["max_amount_egp"])
 
print(f"✅ Feature matrix built: {features_df.shape[0]} customers × {features_df.shape[1]} columns")
display(features_df.head())
 
# Final features for modeling
FEATURE_COLS = [
    "log_total_amount",     # حجم التعامل الكلي
    "log_avg_amount",       # متوسط حجم الفاتورة
    "log_max_amount",       # أكبر فاتورة
    "total_invoices",       # عدد الفواتير
    "paid_ratio",           # معدل الدفع الكامل
    "partial_ratio",        # معدل الدفع الجزئي
    "overdue_ratio",        # معدل التأخر
    "collection_rate",      # معدل التحصيل الفعلي
    "avg_payment_terms",    # متوسط أيام السداد
    "cancel_ratio",         # معدل الإلغاء
    "amount_volatility",    # تباين المبالغ
    "recency_days",         # قِدَم آخر فاتورة
    "tenure_days",          # طول العلاقة التجارية
    "currency_diversity",   # تنوع العملات
    "has_reversal",         # وجود فواتير معكوسة
    "is_supplier",          # مورد أم عميل
    "sent_ratio",           # نسبة الفواتير المُرسَلة
]
 
model_df = features_df[["partner_id", "partner_name"] + FEATURE_COLS].copy()
 
# Handle any remaining NaN
model_df[FEATURE_COLS] = model_df[FEATURE_COLS].fillna(0)
 
print("📋 Feature Summary:")
print(f"  • Customers : {len(model_df):,}")
print(f"  • Features  : {len(FEATURE_COLS)}")
print()
display(model_df[FEATURE_COLS].describe().T.style.background_gradient(cmap="Blues"))
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC ### 3.2 — Feature Distributions
 
# COMMAND ----------
 
fig, axes = plt.subplots(4, 5, figsize=(22, 16))
axes = axes.flatten()
fig.suptitle("Feature Distributions (Customer Level)", fontsize=16, fontweight="bold")
 
for i, col in enumerate(FEATURE_COLS):
    axes[i].hist(model_df[col], bins=25, color="#3498db", edgecolor="white", alpha=0.8)
    axes[i].set_title(col, fontsize=9, fontweight="bold")
    axes[i].set_xlabel("")
    axes[i].tick_params(labelsize=7)
 
# Hide extra subplots
for j in range(len(FEATURE_COLS), len(axes)):
    axes[j].set_visible(False)
 
plt.tight_layout()
plt.show()
 

##  Preprocessing

In [0]:

# ─── Define random state for reproducibility ──────────────────────────────────
RANDOM_STATE = 42

# ─── Scale Features ────────────────────────────────────────────────────────────
# RobustScaler أفضل مع الـ outliers في بيانات الفواتير
scaler = RobustScaler()
X_scaled = scaler.fit_transform(model_df[FEATURE_COLS])
X_scaled_df = pd.DataFrame(X_scaled, columns=FEATURE_COLS, index=model_df.index)
 
print("✅ Features scaled with RobustScaler")
print(f"   Scaled matrix shape: {X_scaled.shape}")
 
# ─── PCA for Visualization ────────────────────────────────────────────────────
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
 
print(f"\n📊 PCA Explained Variance:")
print(f"   PC1: {pca.explained_variance_ratio_[0]*100:.1f}%")
print(f"   PC2: {pca.explained_variance_ratio_[1]*100:.1f}%")
print(f"   Total: {pca.explained_variance_ratio_.sum()*100:.1f}%")
 
# COMMAND ----------
 
# PCA Biplot — Feature Contributions
pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_scaled)
 
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(range(len(pca_full.explained_variance_ratio_)),
       np.cumsum(pca_full.explained_variance_ratio_) * 100,
       color="#3498db", alpha=0.6, label="Cumulative")
ax.bar(range(len(pca_full.explained_variance_ratio_)),
       pca_full.explained_variance_ratio_ * 100,
       color="#e74c3c", alpha=0.8, label="Individual")
ax.axhline(80, color="green", linestyle="--", linewidth=1.5, label="80% threshold")
ax.axhline(95, color="orange", linestyle="--", linewidth=1.5, label="95% threshold")
ax.set_xlabel("Principal Component")
ax.set_ylabel("Explained Variance (%)")
ax.set_title("PCA — Explained Variance per Component", fontweight="bold")
ax.legend()
ax.set_xlim(-0.5, 10)
plt.tight_layout()
plt.show()

##  Optimal K Selection

In [0]:

K_RANGE = range(2, 10)
 
results = {
    "k":           [],
    "inertia":     [],
    "silhouette":  [],
    "davies_bouldin": [],
    "calinski":    [],
}
 
print("🔍 Evaluating K values...")
for k in K_RANGE:
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20, max_iter=500)
    labels = kmeans.fit_predict(X_scaled)
 
    results["k"].append(k)
    results["inertia"].append(kmeans.inertia_)
    results["silhouette"].append(silhouette_score(X_scaled, labels))
    results["davies_bouldin"].append(davies_bouldin_score(X_scaled, labels))
    results["calinski"].append(calinski_harabasz_score(X_scaled, labels))
 
    print(f"  K={k} | Silhouette={results['silhouette'][-1]:.3f} | "
          f"DB={results['davies_bouldin'][-1]:.3f} | Inertia={results['inertia'][-1]:,.0f}")
 
results_df = pd.DataFrame(results)
print("\n✅ K evaluation complete")
 
# COMMAND ----------
 
# Plot all metrics
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Optimal K Selection — Clustering Metrics", fontsize=15, fontweight="bold")
 
# Elbow
axes[0, 0].plot(results_df["k"], results_df["inertia"], "o-", color="#e74c3c", linewidth=2.5, markersize=8)
axes[0, 0].set_title("Elbow Method (Inertia)", fontweight="bold")
axes[0, 0].set_xlabel("Number of Clusters (K)")
axes[0, 0].set_ylabel("Inertia (WCSS)")
axes[0, 0].grid(alpha=0.3)
 
# Silhouette
best_sil_k = results_df.loc[results_df["silhouette"].idxmax(), "k"]
axes[0, 1].plot(results_df["k"], results_df["silhouette"], "o-", color="#2ecc71", linewidth=2.5, markersize=8)
axes[0, 1].axvline(best_sil_k, color="red", linestyle="--", alpha=0.7, label=f"Best K={best_sil_k}")
axes[0, 1].set_title("Silhouette Score (Higher = Better)", fontweight="bold")
axes[0, 1].set_xlabel("Number of Clusters (K)")
axes[0, 1].set_ylabel("Silhouette Score")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)
 
# Davies-Bouldin
best_db_k = results_df.loc[results_df["davies_bouldin"].idxmin(), "k"]
axes[1, 0].plot(results_df["k"], results_df["davies_bouldin"], "o-", color="#3498db", linewidth=2.5, markersize=8)
axes[1, 0].axvline(best_db_k, color="red", linestyle="--", alpha=0.7, label=f"Best K={best_db_k}")
axes[1, 0].set_title("Davies-Bouldin Score (Lower = Better)", fontweight="bold")
axes[1, 0].set_xlabel("Number of Clusters (K)")
axes[1, 0].set_ylabel("DB Score")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)
 
# Calinski-Harabasz
best_ch_k = results_df.loc[results_df["calinski"].idxmax(), "k"]
axes[1, 1].plot(results_df["k"], results_df["calinski"], "o-", color="#9b59b6", linewidth=2.5, markersize=8)
axes[1, 1].axvline(best_ch_k, color="red", linestyle="--", alpha=0.7, label=f"Best K={best_ch_k}")
axes[1, 1].set_title("Calinski-Harabasz Score (Higher = Better)", fontweight="bold")
axes[1, 1].set_xlabel("Number of Clusters (K)")
axes[1, 1].set_ylabel("CH Score")
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)
 
plt.tight_layout()
plt.savefig("/tmp/k_selection.png", dpi=150, bbox_inches="tight")
plt.show()
 
print(f"\n📌 Metric Recommendations:")
print(f"   Best K by Silhouette  : K = {best_sil_k}")
print(f"   Best K by Davies-Bouldin: K = {best_db_k}")
print(f"   Best K by Calinski-Harabasz: K = {best_ch_k}")
 

## Final Model — K-Means Clustering

In [0]:

# ─── Choose K (adjust based on metric results above) ──────────────────────────
# Default = 4 segments (makes business sense: Premium, Regular, At-Risk, Dormant)
# Override here if metrics suggest different K
OPTIMAL_K = 4  # ← غيّر هنا بناءً على نتائج المقاييس فوق
 
print(f"🎯 Selected K = {OPTIMAL_K}")
 
# ─── Train Final Model ────────────────────────────────────────────────────────
with mlflow.start_run(run_name=f"kmeans_k{OPTIMAL_K}_customer_segmentation") as run:
 
    final_model = KMeans(
        n_clusters=OPTIMAL_K,
        random_state=RANDOM_STATE,
        n_init=50,
        max_iter=1000,
        algorithm="lloyd"
    )
    labels = final_model.fit_predict(X_scaled)
 
    # ─── Metrics ──────────────────────────────────────────────────────────────
    sil_score   = silhouette_score(X_scaled, labels)
    db_score    = davies_bouldin_score(X_scaled, labels)
    ch_score    = calinski_harabasz_score(X_scaled, labels)
    inertia     = final_model.inertia_
 
    # ─── Log to MLflow ────────────────────────────────────────────────────────
    mlflow.log_param("k",              OPTIMAL_K)
    mlflow.log_param("n_init",         50)
    mlflow.log_param("max_iter",       1000)
    mlflow.log_param("scaler",         "RobustScaler")
    mlflow.log_param("n_customers",    len(model_df))
    mlflow.log_param("n_features",     len(FEATURE_COLS))
    mlflow.log_param("features",       ", ".join(FEATURE_COLS))
 
    mlflow.log_metric("silhouette_score",         sil_score)
    mlflow.log_metric("davies_bouldin_score",     db_score)
    mlflow.log_metric("calinski_harabasz_score",  ch_score)
    mlflow.log_metric("inertia",                  inertia)
 
    # Log artifacts
    mlflow.log_artifact("/tmp/eda_overview.png",  "plots")
    mlflow.log_artifact("/tmp/k_selection.png",   "plots")
 
    # Log model
    signature = infer_signature(X_scaled, labels)
    mlflow.sklearn.log_model(
        final_model,
        artifact_path="model",
        signature=signature,
        registered_model_name="customer_segmentation_kmeans"
    )
 
    RUN_ID = run.info.run_id
 
print(f"\n✅ Model trained and logged to MLflow")
print(f"   Run ID       : {RUN_ID}")
print(f"   Silhouette   : {sil_score:.4f}  (target > 0.3)")
print(f"   Davies-Bouldin: {db_score:.4f}  (target < 1.5)")
print(f"   Calinski-H   : {ch_score:.2f}")
print(f"   Inertia      : {inertia:,.2f}")
 
# ─── Add cluster labels to dataframe ──────────────────────────────────────────
model_df = model_df.copy()
model_df["cluster_raw"] = labels
model_df["pca_x"] = X_pca[:, 0]
model_df["pca_y"] = X_pca[:, 1]

## Segment Naming & Profiling

In [0]:

# ─── Cluster Profiles ────────────────────────────────────────────────────────
cluster_profile = (
    model_df.groupby("cluster_raw")[FEATURE_COLS]
    .mean()
    .round(3)
)
 
print("📊 Raw Cluster Profiles (Mean Values):")
display(cluster_profile)
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC ### 7.1 — تسمية الـ Segments تلقائياً
# MAGIC
# MAGIC بناءً على خصائص كل Cluster، بنعطيه اسم تجاري يعكس طبيعة العميل.
 
# COMMAND ----------
 
def assign_segment_name(row):
    """
    قاعدة تسمية ذكية بناءً على سلوك الدفع وحجم التعامل.
    يمكن تعديل الحدود بناءً على نتائج بياناتك الفعلية.
    """
    collection = row["collection_rate"]
    volume     = row["log_total_amount"]
    invoices   = row["total_invoices"]
    overdue    = row["overdue_ratio"]
    recency    = row["recency_days"]
 
    avg_collection = cluster_profile["collection_rate"].mean()
    avg_volume     = cluster_profile["log_total_amount"].mean()
    avg_invoices   = cluster_profile["total_invoices"].mean()
 
    if collection >= avg_collection * 1.2 and volume >= avg_volume:
        return "🏆 Premium — High Value & Reliable"
    elif collection >= avg_collection and invoices >= avg_invoices:
        return "✅ Active — Regular & Consistent"
    elif overdue >= 0.7 or collection < avg_collection * 0.6:
        return "⚠️ At-Risk — Frequent Delays"
    elif recency > 180 or invoices <= 1:
        return "💤 Dormant — Low Engagement"
    else:
        return "🔄 Developing — Growing Relationship"
 
 
# Apply naming
segment_map = {
    k: assign_segment_name(cluster_profile.loc[k])
    for k in cluster_profile.index
}
 
model_df["segment"] = model_df["cluster_raw"].map(segment_map)
 
print("🏷️ Segment Assignments:")
for k, name in segment_map.items():
    count = (model_df["cluster_raw"] == k).sum()
    pct   = count / len(model_df) * 100
    print(f"   Cluster {k} → {name} | {count} customers ({pct:.1f}%)")
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC ### 7.2 — Segment Size Overview
 
# COMMAND ----------
 
seg_counts = model_df["segment"].value_counts()
 
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Customer Segment Distribution", fontsize=14, fontweight="bold")
 
colors = ["#2ecc71", "#3498db", "#e74c3c", "#f39c12", "#9b59b6"][:OPTIMAL_K]
 
# Pie
axes[0].pie(seg_counts.values, labels=seg_counts.index, autopct="%1.1f%%",
            colors=colors, startangle=90,
            wedgeprops=dict(edgecolor="white", linewidth=2))
axes[0].set_title("Segment Share (%)", fontweight="bold")
 
# Bar
axes[1].bar(range(len(seg_counts)), seg_counts.values, color=colors, edgecolor="white", linewidth=1.5)
axes[1].set_xticks(range(len(seg_counts)))
axes[1].set_xticklabels([s[:25] + "..." if len(s) > 25 else s for s in seg_counts.index],
                         rotation=15, ha="right", fontsize=9)
axes[1].set_ylabel("Number of Customers")
axes[1].set_title("Customer Count per Segment", fontweight="bold")
for i, v in enumerate(seg_counts.values):
    axes[1].text(i, v + 0.3, str(v), ha="center", fontweight="bold")
 
plt.tight_layout()
plt.show()

## Cluster Visualization

In [0]:

# MAGIC ### 8.1 — PCA 2D Scatter
 
# COMMAND ----------
 
fig, ax = plt.subplots(figsize=(12, 8))
 
unique_segments = model_df["segment"].unique()
colors_map = dict(zip(unique_segments, colors))
 
for seg in unique_segments:
    mask = model_df["segment"] == seg
    ax.scatter(
        model_df.loc[mask, "pca_x"],
        model_df.loc[mask, "pca_y"],
        label=seg,
        color=colors_map[seg],
        alpha=0.75,
        s=100,
        edgecolors="white",
        linewidths=0.5
    )
 
# Annotate top partners
top_n = model_df.nlargest(5, "log_total_amount")
for _, row in top_n.iterrows():
    ax.annotate(row["partner_name"][:20],
                (row["pca_x"], row["pca_y"]),
                fontsize=7, alpha=0.8,
                xytext=(5, 5), textcoords="offset points")
 
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
ax.set_title("Customer Segments — PCA 2D Projection", fontsize=14, fontweight="bold")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
ax.grid(alpha=0.2)
 
plt.tight_layout()
plt.savefig("/tmp/pca_clusters.png", dpi=150, bbox_inches="tight")
plt.show()
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC ### 8.2 — Interactive Plotly Scatter
 
# COMMAND ----------
 
fig_plotly = px.scatter(
    model_df,
    x="pca_x", y="pca_y",
    color="segment",
    hover_name="partner_name",
    hover_data={
        "pca_x": False, "pca_y": False,
        "total_invoices": True,
        "paid_ratio":     ":.2f",
        "collection_rate":":.2f",
        "log_total_amount": False,
    },
    title="🗂️ Customer Segments — Interactive PCA View",
    labels={"pca_x": "PC1", "pca_y": "PC2"},
    color_discrete_sequence=colors,
    template="plotly_white",
    width=900, height=600
)
fig_plotly.update_traces(marker=dict(size=10, opacity=0.8, line=dict(width=0.5, color="white")))
fig_plotly.show()
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC ### 8.3 — Radar Chart per Segment
 
# COMMAND ----------
 
# Normalize features to 0-1 for radar chart
radar_features = [
    "paid_ratio", "collection_rate", "log_total_amount",
    "total_invoices", "overdue_ratio", "amount_volatility",
    "recency_days", "avg_payment_terms"
]
 
radar_data = cluster_profile[radar_features].copy()
radar_normalized = (radar_data - radar_data.min()) / (radar_data.max() - radar_data.min() + 1e-9)
 
fig = go.Figure()
 
for idx, (cluster_id, row) in enumerate(radar_normalized.iterrows()):
    seg_name = segment_map[cluster_id]
    values = row.tolist() + [row.tolist()[0]]  # Close the radar
 
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=radar_features + [radar_features[0]],
        fill="toself",
        name=seg_name[:30],
        line_color=colors[idx],
        opacity=0.7
    ))
 
fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title="📡 Segment Radar Chart — Normalized Feature Profiles",
    showlegend=True,
    template="plotly_white",
    width=800, height=600
)
fig.show()
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC ### 8.4 — Heatmap: Segment vs. Feature (Normalized)
 
# COMMAND ----------
 
fig, ax = plt.subplots(figsize=(14, 5))
 
# Use meaningful subset for heatmap
heatmap_features = [
    "paid_ratio", "overdue_ratio", "collection_rate",
    "log_total_amount", "total_invoices", "avg_payment_terms",
    "cancel_ratio", "amount_volatility", "recency_days",
    "currency_diversity", "sent_ratio"
]
 
heatmap_data = cluster_profile[heatmap_features].copy()
heatmap_norm = (heatmap_data - heatmap_data.min()) / (heatmap_data.max() - heatmap_data.min() + 1e-9)
heatmap_norm.index = [segment_map[i][:35] for i in heatmap_norm.index]
 
sns.heatmap(
    heatmap_norm,
    annot=cluster_profile[heatmap_features].round(2),
    fmt=".2f",
    cmap="RdYlGn",
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "Normalized Value"}
)
ax.set_title("Segment Feature Heatmap (Color = Normalized | Value = Actual Mean)", fontweight="bold")
ax.set_xlabel("Feature")
ax.tick_params(axis="x", rotation=30)
ax.tick_params(axis="y", rotation=0)
plt.tight_layout()
plt.savefig("/tmp/segment_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

##  Deep Segment Profiling

In [0]:

# MAGIC ### 9.1 — Payment Behavior per Segment
 
# COMMAND ----------
 
# Merge segment info with original amounts for visualization
analysis_df = model_df[["partner_id", "segment"]].merge(
    features_df[["partner_id", "total_amount_egp", "avg_amount_egp", "total_invoices"]],
    on="partner_id"
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Payment Behavior by Segment", fontsize=14, fontweight="bold")
 
# Paid Ratio
seg_paid = model_df.groupby("segment")["paid_ratio"].mean().sort_values(ascending=True)
axes[0].barh(seg_paid.index, seg_paid.values, color="#2ecc71", edgecolor="white")
axes[0].set_title("Avg Paid Ratio", fontweight="bold")
axes[0].set_xlabel("Ratio (0–1)")
for i, v in enumerate(seg_paid.values):
    axes[0].text(v + 0.01, i, f"{v:.2f}", va="center", fontsize=9)
 
# Collection Rate
seg_col = model_df.groupby("segment")["collection_rate"].mean().sort_values(ascending=True)
axes[1].barh(seg_col.index, seg_col.values, color="#3498db", edgecolor="white")
axes[1].set_title("Avg Collection Rate", fontweight="bold")
axes[1].set_xlabel("Rate (0–1)")
for i, v in enumerate(seg_col.values):
    axes[1].text(v + 0.01, i, f"{v:.2f}", va="center", fontsize=9)
 
# Overdue Ratio
seg_ov = model_df.groupby("segment")["overdue_ratio"].mean().sort_values(ascending=False)
axes[2].barh(seg_ov.index, seg_ov.values, color="#e74c3c", edgecolor="white")
axes[2].set_title("Avg Overdue Ratio", fontweight="bold")
axes[2].set_xlabel("Ratio (0–1)")
for i, v in enumerate(seg_ov.values):
    axes[2].text(v + 0.01, i, f"{v:.2f}", va="center", fontsize=9)
 
for ax in axes:
    ax.tick_params(axis="y", labelsize=8)
    ax.grid(axis="x", alpha=0.3)
 
plt.tight_layout()
plt.show()
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC ### 9.2 — Volume per Segment
 
# COMMAND ----------
 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Transaction Volume by Segment", fontsize=14, fontweight="bold")
 
seg_vol = analysis_df.groupby("segment")["total_amount_egp"].sum().sort_values(ascending=False)
seg_cnt = analysis_df.groupby("segment")["total_invoices"].sum().sort_values(ascending=False)
 
axes[0].bar(range(len(seg_vol)), seg_vol.values / 1e6, color=colors, edgecolor="white")
axes[0].set_xticks(range(len(seg_vol)))
axes[0].set_xticklabels([s[:20] for s in seg_vol.index], rotation=15, ha="right", fontsize=8)
axes[0].set_ylabel("Total Amount (Millions EGP)")
axes[0].set_title("Total Invoice Value per Segment", fontweight="bold")
 
axes[1].bar(range(len(seg_cnt)), seg_cnt.values, color=colors, edgecolor="white")
axes[1].set_xticks(range(len(seg_cnt)))
axes[1].set_xticklabels([s[:20] for s in seg_cnt.index], rotation=15, ha="right", fontsize=8)
axes[1].set_ylabel("Total Invoices")
axes[1].set_title("Total Invoice Count per Segment", fontweight="bold")
 
plt.tight_layout()
plt.show()
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC ### 9.3 — Top Customers per Segment
 
# COMMAND ----------
 
print("=" * 80)
print("🏅 TOP CUSTOMERS PER SEGMENT")
print("=" * 80)
 
for seg in analysis_df["segment"].unique():
    seg_df = analysis_df[analysis_df["segment"] == seg].copy()
    top5 = seg_df.nlargest(5, "total_amount_egp")[["partner_id"]].merge(
        features_df[["partner_id", "partner_name", "total_invoices", "paid_ratio", "collection_rate", "total_amount_egp"]],
        on="partner_id"
    )[
        ["partner_name", "total_invoices", "paid_ratio", "collection_rate", "total_amount_egp"]
    ]
    print(f"\n{seg}")
    print("-" * 60)
    display(top5.style.format({
        "paid_ratio":       "{:.1%}",
        "collection_rate":  "{:.1%}",
        "total_amount_egp": "{:,.0f} EGP",
    }))
 

##  Business Recommendations per Segment

In [0]:

recommendations = {
    "🏆 Premium — High Value & Reliable": {
        "description":    "عملاء ذوو قيمة تجارية عالية وسجل دفع ممتاز",
        "actions":        [
            "منحهم شروط دفع أفضل وأطول أجل استحقاق",
            "تفضيل في التوريد وخصومات حجم",
            "تعيين Key Account Manager مخصص",
            "فرصة cross-sell وupsell",
        ],
        "kpi_target":    "الحفاظ على Collection Rate > 90%",
        "risk":          "منخفض",
        "priority":      "🟢 High — Retain & Grow"
    },
    "✅ Active — Regular & Consistent": {
        "description":    "عملاء منتظمون بمستوى دفع مقبول",
        "actions":        [
            "متابعة دورية لضمان الالتزام",
            "حوافز للترقية إلى Premium (خصومات عند الدفع المبكر)",
            "إرسال فواتير مبكراً لتجنب التأخير",
        ],
        "kpi_target":    "رفع Collection Rate بنسبة 10%",
        "risk":          "متوسط",
        "priority":      "🔵 Medium — Nurture & Develop"
    },
    "⚠️ At-Risk — Frequent Delays": {
        "description":    "عملاء بتاريخ تأخر متكرر في الدفع",
        "actions":        [
            "تشديد شروط الدفع (دفع مقدم جزئي)",
            "متابعة يومية من فريق التحصيل",
            "تقليص حد الائتمان حتى التسوية",
            "إشعارات تلقائية قبل موعد الاستحقاق بـ 7 أيام",
        ],
        "kpi_target":    "خفض Overdue Ratio إلى < 40%",
        "risk":          "عالي",
        "priority":      "🔴 Urgent — Recover & Protect"
    },
    "💤 Dormant — Low Engagement": {
        "description":    "عملاء غير نشطين أو ذوو حجم تعامل منخفض جداً",
        "actions":        [
            "حملة تنشيط بعروض خاصة",
            "التحقق من استمرار نشاطهم التجاري",
            "إغلاق حسابات غير نشطة لتوفير موارد المتابعة",
        ],
        "kpi_target":    "تحويل 20% منهم إلى Active",
        "risk":          "منخفض",
        "priority":      "⚪ Low — Re-engage or Archive"
    },
    "🔄 Developing — Growing Relationship": {
        "description":    "عملاء جدد أو في طور بناء العلاقة",
        "actions":        [
            "متابعة قريبة في الأشهر الأولى",
            "بناء علاقة ثقة بخدمة عملاء مخصصة",
            "مراقبة مؤشرات الدفع عن كثب",
        ],
        "kpi_target":    "الترقية إلى Active خلال 6 أشهر",
        "risk":          "متوسط",
        "priority":      "🟡 Medium — Monitor & Guide"
    }
}
 
for seg_name, rec in recommendations.items():
    if any(seg_name in s for s in model_df["segment"].unique()):
        print(f"\n{'=' * 70}")
        print(f"  {seg_name}")
        print(f"{'=' * 70}")
        print(f"  📝 Description : {rec['description']}")
        print(f"  🎯 KPI Target  : {rec['kpi_target']}")
        print(f"  ⚡ Risk Level  : {rec['risk']}")
        print(f"  🚦 Priority    : {rec['priority']}")
        print(f"  ✅ Actions:")
        for action in rec["actions"]:
            print(f"      • {action}")
 

##  Save Results

In [0]:

# ─── Prepare output dataframe ─────────────────────────────────────────────────
# Merge model_df with features_df to get original amount columns
output_df = model_df.merge(
    features_df[["partner_id", "total_amount_egp", "avg_amount_egp"]],
    on="partner_id"
)[[
    "partner_id", "partner_name", "cluster_raw", "segment",
    "total_invoices", "total_amount_egp", "avg_amount_egp",
    "paid_ratio", "partial_ratio", "overdue_ratio",
    "collection_rate", "avg_payment_terms",
    "cancel_ratio", "amount_volatility",
    "recency_days", "tenure_days",
    "currency_diversity", "has_reversal",
    "is_supplier", "sent_ratio",
    "pca_x", "pca_y"
]].copy()
 
output_df["model_run_id"]       = RUN_ID
output_df["segmentation_date"]  = pd.Timestamp.today().date()
output_df["optimal_k"]          = OPTIMAL_K
output_df["silhouette_score"]   = round(sil_score, 4)
 
print("📋 Output Schema Preview:")
print(output_df.dtypes)
print(f"\nRows to save: {len(output_df):,}")
display(output_df)
 
# COMMAND ----------
 
# Convert to Spark and save as Delta table
output_spark = spark.createDataFrame(output_df)
 
OUTPUT_TABLE = "smart_odoo.gold.ml_customer_segments"
 
(output_spark
 .write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(OUTPUT_TABLE)
)
 
print(f"✅ Results saved to: {OUTPUT_TABLE}")
print(f"   Rows written: {output_spark.count():,}")
 
# Verify
spark.sql(f"SELECT segment, COUNT(*) as count FROM {OUTPUT_TABLE} GROUP BY segment ORDER BY count DESC").show(truncate=False)
 